# 4.4. Softmax Regression Implementation from Scratch

이번 장에서는 Fashion-MNIST 이미지 분류 모델을 직접 구현한다.

- Softmax 함수
- Weight와 bias
- Forward 계산
- Cross-Entropy Loss
- Accuracy 계산
- Training loop
- Validation
- 예측 결과 확인

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [3]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Fashion-MNIST 데이터 준비

Fashion-MNIST 이미지는 다음 shape을 가진다.

    [channel, height, width]
    [1, 28, 28]

흑백 이미지이므로 channel은 1이다.

Softmax regression에서는 이미지를 한 줄로 펼쳐서 사용한다.

    1 × 28 × 28 = 784

따라서 이미지 한 장은 784개의 feature를 가진 데이터가 된다.

Fashion-MNIST의 class는 10개이므로 모델은 각 이미지마다
10개의 class 점수를 출력해야 한다.

    입력 feature 수 = 784
    출력 class 수 = 10

In [4]:
transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root="./data",
    train=True,
    transform=transform,
    download=True
)

test_dataset = datasets.FashionMNIST(
    root="./data",
    train=False,
    transform=transform,
    download=True
)

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

print("훈련 데이터 개수:", len(train_dataset))
print("테스트 데이터 개수:", len(test_dataset))

훈련 데이터 개수: 60000
테스트 데이터 개수: 10000


In [5]:
X, y = next(iter(train_loader))

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([256, 1, 28, 28])
y shape: torch.Size([256])


현재 minibatch의 shape은 다음과 같다.

    X.shape = [256, 1, 28, 28]
    y.shape = [256]

각 숫자의 의미는 이렇다

    [batch_size, channel, height, width]

X에는 256장의 흑백 이미지가 들어 있다.

각 이미지의 정답 class 번호는 y에 들어 있다.

In [6]:
X_flat = X.reshape(X.shape[0], -1)

print("펼치기 전:", X.shape)
print("펼친 후:", X_flat.shape)

펼치기 전: torch.Size([256, 1, 28, 28])
펼친 후: torch.Size([256, 784])


`reshape(X.shape[0], -1)`을 사용하면 각 이미지를 한 줄로 펼칠 수 있다.

    [256, 1, 28, 28]
            ↓
    [256, 784]

- 256: minibatch에 들어 있는 이미지 개수
- 784: 이미지 한 장의 전체 픽셀 수

`-1`은 나머지 차원의 크기를 PyTorch가 자동으로 계산하라는 뜻이다.